In [18]:
# ============================================================
# Cell 1 — Scenario Configuration (2026–2050)
# ============================================================

from pathlib import Path

# ------------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought")

# ------------------------------------------------------------------
# TIME RANGE
# ------------------------------------------------------------------

START_YEAR = 2026
END_YEAR = 2050

YEARS = [str(y) for y in range(START_YEAR, END_YEAR + 1)]

print("Time span:", START_YEAR, "to", END_YEAR)

# ------------------------------------------------------------------
# SCENARIOS
# ------------------------------------------------------------------

# Recommended operational set
SCENARIOS = [
    "ssp245",   # normal / baseline
    "ssp370",   # high risk
    "ssp585"    # extreme
]

print("Scenarios:", SCENARIOS)

# ------------------------------------------------------------------
# VARIABLES (mapped to your existing model inputs)
# ------------------------------------------------------------------

VARIABLES = {
    "tas": "temperature (t2m equivalent)",
    "huss": "specific humidity (d2m proxy)",
    "pr": "precipitation",
    "mrsos": "soil moisture (upper soil column)",
    "rsds": "surface shortwave radiation",
}

print("Variables:", list(VARIABLES.keys()))

# ------------------------------------------------------------------
# MODEL SELECTION
# ------------------------------------------------------------------

# Start with one stable global model
CLIMATE_MODEL = "MPI-ESM1-2-LR"

print("Climate model:", CLIMATE_MODEL)

# ------------------------------------------------------------------
# DATA OUTPUT ROOT
# ------------------------------------------------------------------

SCENARIO_ROOT = (
    PROJECT_ROOT
    / "data"
    / "scenarios"
)

SCENARIO_ROOT.mkdir(parents=True, exist_ok=True)

print("Scenario root:", SCENARIO_ROOT)

# ------------------------------------------------------------------
# EXPECTED FOLDER STRUCTURE (created later)
# ------------------------------------------------------------------

"""
data/
   scenarios/
       ssp245/
           tas/
           huss/
           mrso/
           rsds/

       ssp370/
           tas/
           huss/
           mrso/
           rsds/

       ssp585/
           tas/
           huss/
           mrso/
           rsds/
"""

Time span: 2026 to 2050
Scenarios: ['ssp245', 'ssp370', 'ssp585']
Variables: ['tas', 'huss', 'pr', 'mrsos', 'rsds']
Climate model: MPI-ESM1-2-LR
Scenario root: C:\Projects\Infer RozviDrought\data\scenarios


'\ndata/\n   scenarios/\n       ssp245/\n           tas/\n           huss/\n           mrso/\n           rsds/\n\n       ssp370/\n           tas/\n           huss/\n           mrso/\n           rsds/\n\n       ssp585/\n           tas/\n           huss/\n           mrso/\n           rsds/\n'

In [19]:
# ============================================================
# Cell 2 — Create Scenario Folder Structure
# ============================================================

from pathlib import Path

# Using variables from Cell 1:
# SCENARIOS
# VARIABLES
# SCENARIO_ROOT

created_paths = []

for scenario in SCENARIOS:
    for var in VARIABLES.keys():

        path = (
            SCENARIO_ROOT
            / scenario
            / var
        )

        path.mkdir(parents=True, exist_ok=True)

        created_paths.append(path)

print("Folders created / verified:")

for p in created_paths:
    print(p)

Folders created / verified:
C:\Projects\Infer RozviDrought\data\scenarios\ssp245\tas
C:\Projects\Infer RozviDrought\data\scenarios\ssp245\huss
C:\Projects\Infer RozviDrought\data\scenarios\ssp245\pr
C:\Projects\Infer RozviDrought\data\scenarios\ssp245\mrsos
C:\Projects\Infer RozviDrought\data\scenarios\ssp245\rsds
C:\Projects\Infer RozviDrought\data\scenarios\ssp370\tas
C:\Projects\Infer RozviDrought\data\scenarios\ssp370\huss
C:\Projects\Infer RozviDrought\data\scenarios\ssp370\pr
C:\Projects\Infer RozviDrought\data\scenarios\ssp370\mrsos
C:\Projects\Infer RozviDrought\data\scenarios\ssp370\rsds
C:\Projects\Infer RozviDrought\data\scenarios\ssp585\tas
C:\Projects\Infer RozviDrought\data\scenarios\ssp585\huss
C:\Projects\Infer RozviDrought\data\scenarios\ssp585\pr
C:\Projects\Infer RozviDrought\data\scenarios\ssp585\mrsos
C:\Projects\Infer RozviDrought\data\scenarios\ssp585\rsds


In [20]:
# ============================================================
# Cell 3 — Safe Download (Skip logic preserved)
# ============================================================

import cdsapi
from pathlib import Path

c = cdsapi.Client()

# Correct CDS variable names confirmed from testing
CDS_VARIABLES = {
    "tas": "near_surface_air_temperature",
    "huss": "near_surface_specific_humidity",
    "pr": "precipitation",
    "mrsos": "moisture_in_upper_portion_of_soil_column",
    "rsds": "surface_downwelling_shortwave_radiation",
}

# Keep skip logic central and easy to change
SKIP_VARIABLES = {
    # Example:
    # "mrsos",
    # "rsds",
}

skipped = []
downloaded = []
failed = []

for scenario in SCENARIOS:
    for var in VARIABLES.keys():

        if var not in CDS_VARIABLES:
            print("Skipping unsupported variable:", var)
            skipped.append((scenario, var, "unsupported_variable"))
            continue

        if var in SKIP_VARIABLES:
            print("Skipping intentionally:", var)
            skipped.append((scenario, var, "intentional_skip"))
            continue

        target_file = (
            SCENARIO_ROOT
            / scenario
            / var
            / f"{var}_{CLIMATE_MODEL}_{scenario}_{START_YEAR}_{END_YEAR}.nc"
        )

        if target_file.exists():
            print("Skipping existing:", target_file.name)
            skipped.append((scenario, var, "already_exists"))
            continue

        print("\nDownloading:", scenario, var)

        try:
            c.retrieve(
                "projections-cmip6",
                {
                    "temporal_resolution": "monthly",
                    "experiment": scenario,
                    "variable": CDS_VARIABLES[var],
                    "model": CLIMATE_MODEL,
                    "year": YEARS,
                    "format": "netcdf",
                },
                str(target_file),
            )

            print("Success:", var)
            downloaded.append((scenario, var, str(target_file)))

        except Exception as e:
            print("Failed:", var)
            print(e)
            failed.append((scenario, var, str(e)))

print("\n===================================")
print("Downloaded:")
for item in downloaded:
    print(item)

print("\n===================================")
print("Skipped:")
for item in skipped:
    print(item)

print("\n===================================")
print("Failed:")
for item in failed:
    print(item)

Skipping existing: tas_MPI-ESM1-2-LR_ssp245_2026_2050.nc
Skipping existing: huss_MPI-ESM1-2-LR_ssp245_2026_2050.nc

Downloading: ssp245 pr


2026-03-21 11:24:51,031 INFO Request ID is d5be9aa2-cf87-4496-9ab8-25d1900aad45
2026-03-21 11:24:51,246 INFO status has been updated to accepted
2026-03-21 11:25:13,266 INFO status has been updated to running
2026-03-21 11:25:42,155 INFO status has been updated to successful


Success: pr

Downloading: ssp245 mrsos


2026-03-21 11:26:04,147 INFO Request ID is 384a9469-74d6-4159-9cf6-28326de4a331
2026-03-21 11:26:04,352 INFO status has been updated to accepted
2026-03-21 11:26:18,593 INFO status has been updated to running
2026-03-21 11:26:39,463 INFO status has been updated to successful


Success: mrsos

Downloading: ssp245 rsds


...

Success: pr

Downloading: ssp585 mrsos


2026-03-21 11:30:34,720 INFO Request ID is c5de5f45-61dd-496d-8df6-faa8f4f38fca
2026-03-21 11:30:34,917 INFO status has been updated to accepted
2026-03-21 11:30:43,877 INFO status has been updated to running
2026-03-21 11:30:56,985 INFO status has been updated to successful


Success: mrsos

Downloading: ssp585 rsds


2026-03-21 11:31:05,510 INFO Request ID is c82fc97c-fad9-4fa6-95bc-b9d243f85562
2026-03-21 11:31:05,711 INFO status has been updated to accepted
2026-03-21 11:31:19,872 INFO status has been updated to running
2026-03-21 11:31:39,254 INFO status has been updated to successful
                                                                                          

Success: rsds

Downloaded:
('ssp245', 'pr', 'C:\\Projects\\Infer RozviDrought\\data\\scenarios\\ssp245\\pr\\pr_MPI-ESM1-2-LR_ssp245_2026_2050.nc')
('ssp245', 'mrsos', 'C:\\Projects\\Infer RozviDrought\\data\\scenarios\\ssp245\\mrsos\\mrsos_MPI-ESM1-2-LR_ssp245_2026_2050.nc')
('ssp245', 'rsds', 'C:\\Projects\\Infer RozviDrought\\data\\scenarios\\ssp245\\rsds\\rsds_MPI-ESM1-2-LR_ssp245_2026_2050.nc')
('ssp370', 'pr', 'C:\\Projects\\Infer RozviDrought\\data\\scenarios\\ssp370\\pr\\pr_MPI-ESM1-2-LR_ssp370_2026_2050.nc')
('ssp370', 'mrsos', 'C:\\Projects\\Infer RozviDrought\\data\\scenarios\\ssp370\\mrsos\\mrsos_MPI-ESM1-2-LR_ssp370_2026_2050.nc')
('ssp370', 'rsds', 'C:\\Projects\\Infer RozviDrought\\data\\scenarios\\ssp370\\rsds\\rsds_MPI-ESM1-2-LR_ssp370_2026_2050.nc')
('ssp585', 'pr', 'C:\\Projects\\Infer RozviDrought\\data\\scenarios\\ssp585\\pr\\pr_MPI-ESM1-2-LR_ssp585_2026_2050.nc')
('ssp585', 'mrsos', 'C:\\Projects\\Infer RozviDrought\\data\\scenarios\\ssp585\\mrsos\\mrsos_MPI-ESM1-2

In [22]:
# ============================================================
# Cell — Save Scenario Download Manifest
# Output: data/artifacts/manifests/scenario_download_manifest_*.json
# ============================================================

import json
from datetime import datetime
from pathlib import Path

# Manifest location
MANIFESTS_DIR = PROJECT_ROOT / "data" / "artifacts" / "manifests"
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

from datetime import datetime, UTC

timestamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")

manifest_path = (
    MANIFESTS_DIR
    / f"scenario_download_manifest_{CLIMATE_MODEL}_{START_YEAR}_{END_YEAR}_{timestamp}.json"
)

records = []

for scenario in SCENARIOS:

    for var in VARIABLES.keys():

        var_dir = SCENARIO_ROOT / scenario / var

        if not var_dir.exists():
            continue

        for file in sorted(var_dir.glob("*.nc")):

            records.append({
                "scenario": scenario,
                "variable": var,
                "model": CLIMATE_MODEL,
                "start_year": START_YEAR,
                "end_year": END_YEAR,
                "file_name": file.name,
                "file_path": str(file),
                "file_size_bytes": file.stat().st_size
            })

manifest = {
    "created_utc": timestamp,
    "project_root": str(PROJECT_ROOT),
    "climate_model": CLIMATE_MODEL,
    "time_span": {
        "start_year": START_YEAR,
        "end_year": END_YEAR
    },
    "scenarios": SCENARIOS,
    "variables": list(VARIABLES.keys()),
    "total_files": len(records),
    "files": records
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

print("===================================")
print("Manifest saved:")
print(manifest_path)
print("Total files recorded:", len(records))

Manifest saved:
C:\Projects\Infer RozviDrought\data\artifacts\manifests\scenario_download_manifest_MPI-ESM1-2-LR_2026_2050_20260321T083902Z.json
Total files recorded: 15


In [28]:
# ============================================================
# Cell — Unzip Scenario Files In Place
# ============================================================

import zipfile
from pathlib import Path

unzipped = []

for scenario in SCENARIOS:
    for var in VARIABLES.keys():

        var_dir = SCENARIO_ROOT / scenario / var

        for file in var_dir.glob("*.nc"):

            with open(file, "rb") as f:
                signature = f.read(2)

            # Check if file is actually a zip
            if signature == b"PK":

                print("Unzipping:", file.name)

                with zipfile.ZipFile(file, "r") as z:
                    z.extractall(var_dir)

                unzipped.append(file)

print("\n===================================")
print("Files unzipped:", len(unzipped))

Unzipping: tas_MPI-ESM1-2-LR_ssp245_2026_2050.nc
Unzipping: huss_MPI-ESM1-2-LR_ssp245_2026_2050.nc
Unzipping: pr_MPI-ESM1-2-LR_ssp245_2026_2050.nc
Unzipping: mrsos_MPI-ESM1-2-LR_ssp245_2026_2050.nc
Unzipping: rsds_MPI-ESM1-2-LR_ssp245_2026_2050.nc
Unzipping: tas_MPI-ESM1-2-LR_ssp370_2026_2050.nc
Unzipping: huss_MPI-ESM1-2-LR_ssp370_2026_2050.nc
Unzipping: pr_MPI-ESM1-2-LR_ssp370_2026_2050.nc
Unzipping: mrsos_MPI-ESM1-2-LR_ssp370_2026_2050.nc
Unzipping: rsds_MPI-ESM1-2-LR_ssp370_2026_2050.nc
Unzipping: tas_MPI-ESM1-2-LR_ssp585_2026_2050.nc
Unzipping: huss_MPI-ESM1-2-LR_ssp585_2026_2050.nc
Unzipping: pr_MPI-ESM1-2-LR_ssp585_2026_2050.nc
Unzipping: mrsos_MPI-ESM1-2-LR_ssp585_2026_2050.nc
Unzipping: rsds_MPI-ESM1-2-LR_ssp585_2026_2050.nc

Files unzipped: 15


In [29]:
# ============================================================
# Cell — Validate Scenario NetCDF Coverage + Grid
# ============================================================

from pathlib import Path
import xarray as xr

summary = []

for scenario in SCENARIOS:
    for var in VARIABLES.keys():

        var_dir = SCENARIO_ROOT / scenario / var
        files = sorted(var_dir.glob("*.nc"))

        if not files:
            summary.append({
                "scenario": scenario,
                "variable": var,
                "status": "missing_file"
            })
            continue

        path = files[0]

        try:

            ds = xr.open_dataset(path, engine="netcdf4")

            data_vars = list(ds.data_vars)

            data_vars = list(ds.data_vars)
            coords = list(ds.coords)
            dims = dict(ds.dims)

            time_dim = dims.get("time", None)
            lat_dim = dims.get("lat", dims.get("latitude", None))
            lon_dim = dims.get("lon", dims.get("longitude", None))

            time_start = str(ds.time.values[0]) if "time" in ds.coords else None
            time_end = str(ds.time.values[-1]) if "time" in ds.coords else None

            summary.append({
                "scenario": scenario,
                "variable": var,
                "status": "ok",
                "file_name": path.name,
                "data_vars": data_vars,
                "coords": coords,
                "time_steps": time_dim,
                "lat_size": lat_dim,
                "lon_size": lon_dim,
                "time_start": time_start,
                "time_end": time_end,
            })

            ds.close()

        except Exception as e:
            summary.append({
                "scenario": scenario,
                "variable": var,
                "status": "failed_to_open",
                "file_name": path.name,
                "error": str(e)
            })

print("===================================")
print("SCENARIO FILE VALIDATION")
print("===================================")

for item in summary:
    print(item)

C:\Users\USER\AppData\Local\Temp\ipykernel_17164\2696125987.py:34: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  dims = dict(ds.dims)
C:\Users\USER\AppData\Local\Temp\ipykernel_17164\2696125987.py:34: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  dims = dict(ds.dims)
C:\Users\USER\AppData\Local\Temp\ipykernel_17164\2696125987.py:34: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  dims = dict(d

SCENARIO FILE VALIDATION
{'scenario': 'ssp245', 'variable': 'tas', 'status': 'ok', 'file_name': 'tas_Amon_MPI-ESM1-2-LR_ssp245_r1i1p1f1_gn_20260116-20501216.nc', 'data_vars': ['time_bnds', 'lat_bnds', 'lon_bnds', 'tas'], 'coords': ['time', 'lat', 'lon', 'height'], 'time_steps': 300, 'lat_size': 96, 'lon_size': 192, 'time_start': '2026-01-16T12:00:00.000000000', 'time_end': '2050-12-16T12:00:00.000000000'}
{'scenario': 'ssp245', 'variable': 'huss', 'status': 'ok', 'file_name': 'huss_Amon_MPI-ESM1-2-LR_ssp245_r1i1p1f1_gn_20260116-20501216.nc', 'data_vars': ['time_bnds', 'lat_bnds', 'lon_bnds', 'huss'], 'coords': ['time', 'lat', 'lon', 'height'], 'time_steps': 300, 'lat_size': 96, 'lon_size': 192, 'time_start': '2026-01-16T12:00:00.000000000', 'time_end': '2050-12-16T12:00:00.000000000'}
{'scenario': 'ssp245', 'variable': 'pr', 'status': 'ok', 'file_name': 'pr_Amon_MPI-ESM1-2-LR_ssp245_r1i1p1f1_gn_20260116-20501216.nc', 'data_vars': ['time_bnds', 'lat_bnds', 'lon_bnds', 'pr'], 'coords': [

C:\Users\USER\AppData\Local\Temp\ipykernel_17164\2696125987.py:34: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  dims = dict(ds.dims)


In [30]:
# ============================================================
# Cell — Build Combined Scenario Dataset
# ============================================================

import xarray as xr
from pathlib import Path

combined = {}

for scenario in SCENARIOS:

    print("\nProcessing:", scenario)

    datasets = []

    for var in VARIABLES.keys():

        file = (
            SCENARIO_ROOT
            / scenario
            / var
        ).glob("*.nc")

        file = list(file)[0]

        ds = xr.open_dataset(file)

        datasets.append(ds)

        print("Loaded:", var)

    merged = xr.merge(datasets)

    combined[scenario] = merged

    print("Merged dataset ready")

print("\n===================================")
print("All scenarios loaded into memory")


Processing: ssp245
Loaded: tas
Loaded: huss
Loaded: pr
Loaded: mrsos
Loaded: rsds
Merged dataset ready

Processing: ssp370


C:\Users\USER\AppData\Local\Temp\ipykernel_17164\1770205975.py:32: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge(datasets)
C:\Users\USER\AppData\Local\Temp\ipykernel_17164\1770205975.py:32: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge(datasets)
C:\Users\USER\AppData\Local\Temp\ipykernel_17164\17702

Loaded: tas
Loaded: huss
Loaded: pr
Loaded: mrsos
Loaded: rsds
Merged dataset ready

Processing: ssp585


C:\Users\USER\AppData\Local\Temp\ipykernel_17164\1770205975.py:32: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge(datasets)
C:\Users\USER\AppData\Local\Temp\ipykernel_17164\1770205975.py:32: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge(datasets)
C:\Users\USER\AppData\Local\Temp\ipykernel_17164\17702

Loaded: tas
Loaded: huss
Loaded: pr
Loaded: mrsos
Loaded: rsds
Merged dataset ready

All scenarios loaded into memory


C:\Users\USER\AppData\Local\Temp\ipykernel_17164\1770205975.py:32: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge(datasets)
C:\Users\USER\AppData\Local\Temp\ipykernel_17164\1770205975.py:32: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  merged = xr.merge(datasets)
C:\Users\USER\AppData\Local\Temp\ipykernel_17164\17702

In [31]:
# ============================================================
# Cell — Save Combined Scenario Datasets
# ============================================================

OUTPUT_ROOT = Path(
    r"C:\Projects\Infer RozviDrought\data\artifacts\datasets"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

saved_files = []

for scenario, ds in combined.items():

    output_file = (
        OUTPUT_ROOT
        / f"scenario_dataset_{scenario}_{START_YEAR}_{END_YEAR}.nc"
    )

    print("\nSaving:", scenario)

    ds.to_netcdf(output_file)

    saved_files.append(output_file)

    print("Saved:", output_file.name)

print("\n===================================")
print("Datasets saved:", len(saved_files))


Saving: ssp245
Saved: scenario_dataset_ssp245_2026_2050.nc

Saving: ssp370
Saved: scenario_dataset_ssp370_2026_2050.nc

Saving: ssp585
Saved: scenario_dataset_ssp585_2026_2050.nc

Datasets saved: 3
